In [1]:
import torch
import torch.nn as nn

# Dataset and Training
# This section adds dataset loading, DataLoader setup, and the training/validation loops for the deepfake detector.
import os
from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, random_split
import cv2
import numpy as np
import torch


# =====================================
# Attention Block
# =====================================
class AttentionBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()

        self.fc = nn.Sequential(
            nn.Linear(channels, channels // 8),
            nn.ReLU(),
            nn.Linear(channels // 8, channels),
            nn.Sigmoid()
        )

    def forward(self, x):

        b, c, h, w = x.size()

        avg_pool = torch.mean(x, dim=[2, 3])

        attention = self.fc(avg_pool)

        attention = attention.view(b, c, 1, 1)

        return x * attention


# =====================================
# FFT Branch
# =====================================
class FFTBranch(nn.Module):
    def __init__(self):
        super().__init__()

        self.cnn = nn.Sequential(

            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d(1)
)

    def forward(self, x):

        gray = torch.mean(x, dim=1, keepdim=True)

        fft = torch.fft.fft2(gray)

        fft = torch.abs(fft)

        fft = torch.log(fft + 1e-6)

        features = self.cnn(fft)

        return features.view(features.size(0), -1)


# =====================================
# Deepfake Detector
# =====================================
class DeepfakeDetector(nn.Module):

    def __init__(self):
        super().__init__()

        # CNN Feature Extractor
        self.cnn = nn.Sequential(

            nn.Conv2d(3, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(256, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # NEW BLOCK
            nn.Conv2d(512, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.attention = AttentionBlock(512)

        self.fft_branch = FFTBranch()

        self.fusion = nn.Sequential(
            nn.Linear(512 + 64, 512),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        self.classifier = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 2)
        )

    def forward(self, x):

        # x shape:
        # (batch, frames, 3, 224, 224)

        b, t, c, h, w = x.shape

        frame_features = []
        fft_features = []

        # Process all frames
        for i in range(t):

            frame = x[:, i]

            # CNN Features
            spatial = self.cnn(frame)

            

            spatial = self.attention(spatial)

            spatial = nn.functional.adaptive_avg_pool2d(
                spatial,
                1
            )

            spatial = spatial.view(b, 512)

            frame_features.append(spatial)

            # FFT Features
            freq = self.fft_branch(frame)
            fft_features.append(freq)

        # Average CNN Features
        spatial = torch.stack(
            frame_features,
            dim=1
        ).mean(dim=1)

        # Average FFT Features
        freq = torch.stack(
           fft_features,
            dim=1,
        ).mean(dim=1)

        

        # Fusion
        fused = torch.cat(
            [spatial, freq],
            dim=1
        )

        fused = self.fusion(fused)

        # Classification
        output = self.classifier(fused)

        return output




# ==============================
# FRAME EXTRACTION
# ==============================
def extract_frames(video_path, num_frames=10):

    cap = cv2.VideoCapture(video_path)

    frames = []

    total_frames = int(
        cap.get(cv2.CAP_PROP_FRAME_COUNT)
    )

    step = max(
        total_frames // num_frames,
        1
    )

    for i in range(num_frames):

        cap.set(
            cv2.CAP_PROP_POS_FRAMES,
            i * step
        )

        success, frame = cap.read()

        if success:

            frame = cv2.cvtColor(
                frame,
                cv2.COLOR_BGR2RGB
            )

            frame = cv2.resize(
                frame,
                (224, 224)
            )

            frame = frame.astype(np.float32)

            frame /= 255.0

            frame = (
                frame -
                np.array([0.485, 0.456, 0.406])
            ) / np.array([
                0.229,
                0.224,
                0.225
            ])

            frame = np.transpose(
                frame,
                (2, 0, 1)
            )

            frames.append(frame)

    cap.release()

    if len(frames) == 0:

        return torch.zeros(
            (
                num_frames,
                3,
                224,
                224
            ),
            dtype=torch.float32,
        )

    while len(frames) < num_frames:
        frames.append(frames[-1])

    return torch.tensor(
        np.array(frames),
        dtype=torch.float32,
    )

class VideoDataset(Dataset):

    def __init__(self, root_dir, num_frames=10):
        self.samples = []
        self.num_frames = num_frames

        for label, folder in enumerate(
            ["real", "fake"]
        ):

            path = os.path.join(
                root_dir,
                folder
            )

            if not os.path.exists(path):
                continue

            for file in os.listdir(path):

                if file.lower().endswith(".mp4"):

                    self.samples.append(
                        (
                            os.path.join(path, file),
                            label
                        )
                    )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        video_path, label = self.samples[idx]

        frames = extract_frames(
            video_path,
            num_frames=self.num_frames
        )

        return frames, torch.tensor(
            label,
            dtype=torch.long
        )

def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for inputs, labels in dataloader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)

        
        
        
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        preds = outputs.argmax(dim=1)
        total_loss += loss.item() * inputs.size(0)
        total_correct += (preds == labels).sum().item()
        total_samples += inputs.size(0)

    avg_loss = total_loss / total_samples if total_samples else 0.0
    avg_acc = total_correct / total_samples if total_samples else 0.0
    return avg_loss, avg_acc


@torch.no_grad()
def validate_epoch(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for inputs, labels in dataloader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        
        loss = criterion(outputs, labels)
        

        preds = outputs.argmax(dim=1)
        total_loss += loss.item() * inputs.size(0)
        total_correct += (preds == labels).sum().item()
        total_samples += inputs.size(0)

    avg_loss = total_loss / total_samples if total_samples else 0.0
    avg_acc = total_correct / total_samples if total_samples else 0.0
    return avg_loss, avg_acc


# Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
root_dir = "dataset"
num_frames = 15
batch_size = 8
learning_rate = 3e-4
num_epochs = 50

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# =====================================
# LOAD  DATASET (500 VIDEOS)
# =====================================

full_dataset = VideoDataset(
    root_dir,
    num_frames=num_frames
)

if len(full_dataset) == 0:
    raise RuntimeError(
        f"No videos found in '{root_dir}/real' or '{root_dir}/fake'."
    )

# Count classes
real_count = 0
fake_count = 0

for _, label in full_dataset:

    if label == 0:
        real_count += 1
    else:
        fake_count += 1

print("Real videos:", real_count)
print("Fake videos:", fake_count)
print("Total videos:", len(full_dataset))

frames, label = full_dataset[0]

print("Frame shape:", frames.shape)
print("First label:", label)

# =====================================
# TRAIN / VALIDATION SPLIT
# =====================================

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

print("Train videos:", len(train_dataset))
print("Validation videos:", len(val_dataset))

# =====================================
# DATALOADERS
# =====================================

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

# Model
model = DeepfakeDetector().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=learning_rate,
    weight_decay=1e-5
)

scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=10,
    gamma=0.5
)
# Training Loop
for epoch in range(1, num_epochs + 1):

    train_loss, train_acc = train_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    val_loss, val_acc = validate_epoch(
        model,
        val_loader,
        criterion,
        device
    )
    scheduler.step()
    print(
        f"Epoch {epoch}/{num_epochs} - "
        f"Train loss: {train_loss:.4f}, "
        f"Train acc: {train_acc:.4f} - "
        f"Val loss: {val_loss:.4f}, "
        f"Val acc: {val_acc:.4f}")

Real videos: 1000
Fake videos: 1000
Total videos: 2000
Frame shape: torch.Size([15, 3, 224, 224])
First label: tensor(0)
Train videos: 1600
Validation videos: 400
Epoch 1/50 - Train loss: 0.6995, Train acc: 0.5050 - Val loss: 0.7008, Val acc: 0.4825
Epoch 2/50 - Train loss: 0.6961, Train acc: 0.4981 - Val loss: 0.6934, Val acc: 0.5050
Epoch 3/50 - Train loss: 0.6951, Train acc: 0.4969 - Val loss: 0.6975, Val acc: 0.4825
Epoch 4/50 - Train loss: 0.6935, Train acc: 0.5162 - Val loss: 0.6935, Val acc: 0.4975
Epoch 5/50 - Train loss: 0.6952, Train acc: 0.4850 - Val loss: 0.6940, Val acc: 0.4825
Epoch 6/50 - Train loss: 0.6952, Train acc: 0.4869 - Val loss: 0.6939, Val acc: 0.4825
Epoch 7/50 - Train loss: 0.6938, Train acc: 0.5169 - Val loss: 0.6932, Val acc: 0.5250
Epoch 8/50 - Train loss: 0.6945, Train acc: 0.5038 - Val loss: 0.6951, Val acc: 0.4825
Epoch 9/50 - Train loss: 0.6938, Train acc: 0.4894 - Val loss: 0.6951, Val acc: 0.4825
Epoch 10/50 - Train loss: 0.6940, Train acc: 0.5050 - 